# Case study: the same aggregation in SQL, pandas and Polars on the availability calendar

*Session 3, block 3 · Author: course team · Licence: CC-BY-4.0*

**Why.** A housing analyst wants a monthly overview for the coming year: in each district, how many listing-nights
are on offer, what share of them is still free, and what share can only be booked for a month or more? The answer
comes from the **availability calendar**: one row per listing and night, 4.7 million rows. That is ten times more
than any other table of the course, and large enough that the choice of tool starts to matter.

**Question.** For each district and calendar month: the number of nights, the number of listings, the share of
nights still free, and the share of nights with a minimum stay of 28 nights or more. The district comes from the
listings table, so the calendar must be joined with it.

We answer the question three times, in **SQL** (DuckDB), **pandas** and **Polars**, on the **raw CSV file** that
Inside Airbnb publishes (`calendar.csv.gz`, 160 MB once unpacked), and compare the **code**, the **runtime** and the
**memory use**. At the end we repeat the timing on the prepared **Parquet** file to see how much the file format
matters.

Your timings will differ from your neighbours': they depend on the processor, the number of cores, the disk and the
library versions. Compare the *ratios*, not the absolute numbers.

In [ ]:
import gzip
import os
import platform
import shutil
import tempfile
import time
from pathlib import Path

import duckdb
import pandas as pd
import polars as pl

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
GZ = ROOT / "case-study" / "data" / "raw" / "airbnb" / "calendar.csv.gz"
LISTINGS = ROOT / "case-study" / "data" / "airbnb" / "listings.parquet"
PARQUET = ROOT / "case-study" / "data" / "airbnb" / "calendar.parquet"
assert GZ.exists(), "run case-study/prepare_airbnb.py first (it keeps the raw files)"

# the CSV file is unpacked once, outside the repository (set AIRBNB_CSV_DIR to choose another folder)
CSV = Path(os.environ.get("AIRBNB_CSV_DIR", Path(tempfile.gettempdir()) / "airbnb-csv")) / "calendar.csv"
if not CSV.exists():
    CSV.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(GZ, "rb") as src, open(CSV, "wb") as dst:
        shutil.copyfileobj(src, dst)
print(f"{CSV.stat().st_size / 1e6:,.0f} MB of CSV in {CSV}; Parquet: {PARQUET.stat().st_size / 1e6:,.1f} MB")
print(platform.platform(), "| cores", os.cpu_count(), "| pandas", pd.__version__, "| polars", pl.__version__,
      "| duckdb", duckdb.__version__)

In [ ]:
# a first look: the columns and three rows
pd.read_csv(CSV, nrows=3)

Everything in a CSV file is text: the dates are strings such as `2026-07-03`, and `available` is `t` or `f`. Each
tool must parse the file and decide on the types. The Parquet file of the course stores the same 4.7 million rows
with types, compressed, in about 2 MB.

## 1. SQL (DuckDB on the CSV file)

In [ ]:
SQL = f"""
SELECT l.district,
       date_trunc('month', c.date)::date AS month,
       COUNT(*) AS nights,
       COUNT(DISTINCT c.listing_id) AS listings,
       AVG(CASE WHEN c.available THEN 1.0 ELSE 0.0 END) AS share_free,
       AVG(CASE WHEN c.minimum_nights >= 28 THEN 1.0 ELSE 0.0 END) AS share_medium_term
FROM read_csv('{CSV}', types = {{'available': 'BOOLEAN', 'date': 'DATE'}}) AS c
JOIN read_parquet('{LISTINGS}') AS l ON l.id = c.listing_id
GROUP BY l.district, month
ORDER BY l.district, month
"""


def run_sql():
    return duckdb.sql(SQL).pl()        # result as a Polars DataFrame


res_sql = run_sql()
res_sql.head()

DuckDB reads the CSV file in parallel and joins it with the listings, which it reads straight from Parquet. The
types are stated in the query (`available` as `BOOLEAN`, `date` as `DATE`); otherwise DuckDB guesses them from a
sample of the file.

## 2. pandas (eager)

pandas executes each line immediately (**eager evaluation**): it reads the whole file with all five columns, builds
the merged table in memory, then the new columns, then the groups.

In [ ]:
def run_pandas():
    calendar = pd.read_csv(CSV, parse_dates=["date"])
    listings = pd.read_parquet(LISTINGS, columns=["id", "district"])
    merged = calendar.merge(listings, left_on="listing_id", right_on="id")      # inner join
    return (
        merged.assign(month=merged["date"].dt.to_period("M").dt.to_timestamp(),
                      free=merged["available"].eq("t"),
                      medium_term=merged["minimum_nights"].ge(28))
        .groupby(["district", "month"], as_index=False)
        .agg(nights=("free", "size"), listings=("listing_id", "nunique"),
             share_free=("free", "mean"), share_medium_term=("medium_term", "mean"))
        .sort_values(["district", "month"])
    )


res_pd = run_pandas()
res_pd.head()

## 3. Polars: eager and lazy

Polars code is built from **expressions** such as `(pl.col("available") == "t").mean()`. With `scan_csv` instead of
`read_csv`, nothing is computed yet: Polars builds a **query plan** (a `LazyFrame`), optimises it and runs it on
`.collect()`. The same query function serves all three variants.

In [ ]:
def build_query(calendar: pl.LazyFrame) -> pl.LazyFrame:
    listings = pl.scan_parquet(LISTINGS).select("id", "district")
    return (
        calendar.join(listings, left_on="listing_id", right_on="id")            # inner join
        .group_by("district", pl.col("date").str.to_date().dt.truncate("1mo").alias("month"))
        .agg(nights=pl.len(),
             listings=pl.col("listing_id").n_unique(),
             share_free=(pl.col("available") == "t").mean(),
             share_medium_term=(pl.col("minimum_nights") >= 28).mean())
        .sort("district", "month")
    )


def run_polars_eager():
    # read everything first, then compute (like pandas)
    return build_query(pl.read_csv(CSV).lazy()).collect()


def run_polars_lazy():
    return build_query(pl.scan_csv(CSV)).collect()


def run_polars_streaming():
    # the streaming engine processes the file in batches; it also works for data larger than memory
    return build_query(pl.scan_csv(CSV)).collect(engine="streaming")


res_pl = run_polars_lazy()
res_pl.head()

### What did the optimiser do?

`explain()` prints the optimised plan. Look for the join and for the scan of the listings: Polars reads only the two
columns it needs from the 42 columns of `listings.parquet` (**projection pushdown**). A filter, for example
`pl.col("district") == "Neukölln"` after the join, would be moved down to the scan of the listings (**predicate
pushdown**), so that fewer rows are joined.

In [ ]:
print(build_query(pl.scan_csv(CSV)).explain())

In [ ]:
# predicate pushdown: the filter on the district appears inside the scan of the listings
print(build_query(pl.scan_csv(CSV)).filter(pl.col("district") == "Neukölln").explain())

## 4. Are the results the same?

Before comparing speed, check that the tools answer the same question. The groups and counts must agree exactly;
averages may differ in the last digits.

In [ ]:
from polars.testing import assert_frame_equal

key = ["district", "month"]
a = res_pl.sort(key)
b = pl.from_pandas(res_pd).with_columns(pl.col("month").cast(pl.Date)).sort(key)
c = res_sql.sort(key)
for other in (b, c):
    assert_frame_equal(a, other, check_dtypes=False, abs_tol=1e-9)
print("all three agree:", a.shape)
print("nights in the result:", a["nights"].sum(), "of", pl.scan_csv(CSV).select(pl.len()).collect().item(), "calendar rows")

The result has fewer nights than the calendar has rows: the inner join drops the 28,835 rows of the 79 listings
that are not in the listings table (block 1). All three tools drop them, so the results agree; a left join in one
tool and an inner join in another would not.

The check also passes because each version says explicitly what counts as "free". The raw file stores `t` and `f`.
DuckDB was told that the column is `BOOLEAN`; pandas and Polars read it as text and compare with `"t"`. A rule such
as `available == True` would silently give a share of 0 % in pandas and Polars, because the text `"t"` is never
equal to `True`:

In [ ]:
sample = pd.read_csv(CSV, nrows=1000)
print("pandas dtype:", sample["available"].dtype, "| share == True:", (sample["available"] == True).mean(),  # noqa: E712
      "| share == 't':", (sample["available"] == "t").mean())
print("Polars dtype:", pl.read_csv(CSV, n_rows=1000)["available"].dtype)

## 5. Runtime

Each version runs three times; we keep the fastest run (the others include warm-up effects such as the operating
system's file cache).

In [ ]:
def best_of(fn, repeat=3):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)


runs = {"SQL (DuckDB)": run_sql, "pandas": run_pandas, "Polars eager": run_polars_eager,
        "Polars lazy": run_polars_lazy, "Polars streaming": run_polars_streaming}
runtime = pd.Series({name: best_of(fn) for name, fn in runs.items()}, name="seconds").round(3)
runtime.to_frame().assign(relative_to_pandas=lambda d: (d["seconds"] / d.loc["pandas", "seconds"]).round(2))

## 6. Memory

**In-memory size of the table.** pandas reports the bytes of each column (`deep=True` includes the strings);
Polars estimates the size of its Arrow buffers. Both hold all five columns of all 4.7 million rows.

In [ ]:
raw_pd = pd.read_csv(CSV)
raw_pd_typed = pd.read_csv(CSV, parse_dates=["date"]).assign(available=lambda d: d["available"].eq("t"))
raw_pl = pl.read_csv(CSV)
print(f"pandas, dates and t/f as text:  {raw_pd.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
print(f"pandas, typed columns:          {raw_pd_typed.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
print(f"Polars, as read:                {raw_pl.estimated_size() / 1e6:,.0f} MB")
print(f"CSV on disk: {CSV.stat().st_size / 1e6:,.0f} MB; gzip: {GZ.stat().st_size / 1e6:,.1f} MB; "
      f"Parquet: {PARQUET.stat().st_size / 1e6:,.1f} MB")
del raw_pd, raw_pd_typed, raw_pl

**Peak memory of the whole computation.** Memory that is freed again does not show up in the table size, so we
measure the **peak resident memory** of a fresh Python process for each version. The baseline is a process that
only imports the libraries.

In [ ]:
import subprocess
import sys
import textwrap

BENCH = textwrap.dedent(f"""
    import resource, sys
    import duckdb, pandas as pd, polars as pl
    CSV, LISTINGS = {str(CSV)!r}, {str(LISTINGS)!r}
    version = sys.argv[1]
    if version == "pandas":
        cal = pd.read_csv(CSV, parse_dates=["date"])
        m = cal.merge(pd.read_parquet(LISTINGS, columns=["id", "district"]), left_on="listing_id", right_on="id")
        out = (m.assign(month=m["date"].dt.to_period("M"), free=m["available"].eq("t"), mt=m["minimum_nights"].ge(28))
                .groupby(["district", "month"])
                .agg(nights=("free", "size"), listings=("listing_id", "nunique"), share_free=("free", "mean"),
                     share_medium_term=("mt", "mean")))
    elif version.startswith("polars"):
        src = pl.read_csv(CSV).lazy() if version == "polars_eager" else pl.scan_csv(CSV)
        q = (src.join(pl.scan_parquet(LISTINGS).select("id", "district"), left_on="listing_id", right_on="id")
                .group_by("district", pl.col("date").str.to_date().dt.truncate("1mo").alias("month"))
                .agg(nights=pl.len(), listings=pl.col("listing_id").n_unique(),
                     share_free=(pl.col("available") == "t").mean(),
                     share_medium_term=(pl.col("minimum_nights") >= 28).mean()))
        out = q.collect(engine="streaming") if version == "polars_streaming" else q.collect()
    elif version == "duckdb":
        out = duckdb.sql({SQL!r}).fetchall()
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    print(peak / (1024 ** 2 if sys.platform == "darwin" else 1024))   # bytes on macOS, KiB on Linux
""")

versions = {"baseline (imports only)": "none", "SQL (DuckDB)": "duckdb", "pandas": "pandas",
            "Polars eager": "polars_eager", "Polars lazy": "polars_lazy", "Polars streaming": "polars_streaming"}
peak = pd.Series({name: float(subprocess.run([sys.executable, "-c", BENCH, v], capture_output=True, text=True,
                                             check=True).stdout) for name, v in versions.items()},
                 name="peak_MB").round(0)
peak.to_frame().assign(above_baseline=lambda d: d["peak_MB"] - d.loc["baseline (imports only)", "peak_MB"])

`ru_maxrss` is the largest amount of main memory the process held at any moment. On Windows, the `resource` module
does not exist; there, skip this cell or use `psutil`.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
runtime.plot.barh(ax=axes[0], color="#2a78d6")
axes[0].set(title="Runtime (best of 3)", xlabel="seconds")
(peak.drop("baseline (imports only)") - peak["baseline (imports only)"]).plot.barh(ax=axes[1], color="#eb6834")
axes[1].set(title="Peak memory above baseline", xlabel="MB")
for ax in axes:
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 7. The same query on Parquet

The prepared file `calendar.parquet` holds the same rows with types: `date` as a date and `available` as true/false.
No text has to be parsed. Only the expressions for these two columns change.

In [ ]:
SQL_PQ = SQL.replace(f"read_csv('{CSV}', types = {{'available': 'BOOLEAN', 'date': 'DATE'}})", f"read_parquet('{PARQUET}')")


def run_pandas_parquet():
    calendar = pd.read_parquet(PARQUET)
    merged = calendar.merge(pd.read_parquet(LISTINGS, columns=["id", "district"]), left_on="listing_id", right_on="id")
    return (merged.assign(month=merged["date"].dt.to_period("M").dt.to_timestamp(),
                          medium_term=merged["minimum_nights"].ge(28))
            .groupby(["district", "month"], as_index=False)
            .agg(nights=("available", "size"), listings=("listing_id", "nunique"),
                 share_free=("available", "mean"), share_medium_term=("medium_term", "mean")))


def run_polars_parquet():
    listings = pl.scan_parquet(LISTINGS).select("id", "district")
    return (pl.scan_parquet(PARQUET).join(listings, left_on="listing_id", right_on="id")
            .group_by("district", pl.col("date").dt.truncate("1mo").alias("month"))
            .agg(nights=pl.len(), listings=pl.col("listing_id").n_unique(),
                 share_free=pl.col("available").mean(), share_medium_term=(pl.col("minimum_nights") >= 28).mean())
            .collect())


assert run_polars_parquet().sort(key)["nights"].to_list() == res_pl["nights"].to_list()
runtime_pq = pd.Series({"SQL (DuckDB)": best_of(lambda: duckdb.sql(SQL_PQ).fetchall()),
                        "pandas": best_of(run_pandas_parquet),
                        "Polars lazy": best_of(run_polars_parquet)}, name="seconds_parquet").round(3)
pd.concat([runtime, runtime_pq], axis=1)

**How to read the numbers.** When the course team ran this notebook twice (Apple silicon laptop with 18 cores,
busy with other work; pandas 2.3, Polars 1.44, DuckDB 1.5), the ratios were stable although the absolute times varied
by about a quarter. On the CSV file, pandas needed about 2 to 3 seconds, Polars eager and lazy about 0.6 to 0.7
seconds, Polars streaming and DuckDB about 0.2 seconds: a factor of about 12 between pandas and the fastest tools.
pandas parses the file on one core and builds every intermediate table, including a merged copy of all 4.7 million
rows; DuckDB and the streaming engine of Polars parse in parallel and process the file in batches. Peak memory
followed the same order: about 1.5 GB above the baseline for pandas, 1.0 to 1.1 GB for Polars eager and lazy,
0.55 GB for Polars streaming and 0.24 GB for DuckDB. Here, streaming did save memory, unlike the default lazy engine,
which still holds the whole parsed table. Peak resident memory is an upper bound: it includes memory the operating
system could reclaim. Switching from CSV to Parquet halved the runtime of pandas and Polars lazy, without changing
the library.

For 4.7 million rows, all tools fit easily on a laptop; the differences matter when the table is a hundred times
larger, or when the same step runs every night. Check your own numbers before you repeat claims such as "lazy
always saves memory".

## 8. Compare the code

| Step | SQL | pandas | Polars |
|---|---|---|---|
| read | `FROM read_csv(...)` | `pd.read_csv` (eager) | `pl.scan_csv` (lazy) |
| join | `JOIN ... ON` | `merge(...)` | `join(...)` |
| month | `date_trunc('month', date)` | `dt.to_period("M")` | `dt.truncate("1mo")` |
| share of a condition | `AVG(CASE WHEN ... THEN 1.0 ELSE 0.0 END)` | `eq(...)` then `mean` | `(... == ...).mean()` |
| distinct count | `COUNT(DISTINCT ...)` | `nunique` | `n_unique` |
| group | `GROUP BY` | `groupby` | `group_by` |

**Questions for you**

1. Which version is the easiest to read for you, and which would you trust in a report? Why?
2. Remove the join and group only by `listing_id` and month (no district). Does the ranking of the tools change?
3. Add a filter for July and August 2026. Where does Polars apply it in the plan (`explain()`)? Is pandas faster if
   you filter before the merge?
4. In which situation would you load the calendar into PostgreSQL instead of using any of the three tools on the
   file? Think of a team that updates the data every quarter.

In [ ]:
# space for questions 2-4